In [ ]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split, StratifiedKFold,GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


## set up pyspark session

In [4]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 07:02:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [5]:
# set up config
model_train_date_str = "2024-03-01"
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 3, 1, 0, 0),
 'model_train_date_str': '2024-03-01',
 'oot_end_date': datetime.datetime(2024, 2, 29, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 1, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2023, 12, 31, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 1, 1, 0, 0)}


## get label store

In [6]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [7]:
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted labels_sdf 4037 2023-01-01 00:00:00 2024-02-29 00:00:00


## get features

In [8]:
# connect to feature store
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()

row_count: 11974
+-----------+--------------------+---------------+------+--------+-------------------------+-------------+-------------+---------------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-------------------------+---+-------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------------+-------------------+--------------------+--------------------------+-----------------------+--------------------+--------------+-------------------+-----------------------+-------------------+----------------------------+--------+----------------+----------------+-----------------------+
|Customer_ID|     

In [13]:
# extract features from feature store
features_sdf = features_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted features_sdf", features_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted features_sdf 6961 2023-01-01 00:00:00 2024-02-29 00:00:00


## prepare data for modeling

In [16]:
# prepare data for modeling
data_pdf = labels_sdf.join(features_sdf, on=["Customer_ID", "snapshot_date"], how="left").toPandas()
data_pdf

,Customer_ID,snapshot_date,loan_id,label,label_def,loan_id,loan_start_date,tenure,loan_amt,Scheduled_Installment_Amt,...,Delayed_Payment_Rate,Post_EMI_Slack,Loan_Type_Diversity,Credit_Utilization_Band,Credit_History_Band,Recent_Credit_Expansion_Flag,Age_Band,Occupation_Group,Clickstream_Mean,Clickstream_Vector_Norm
0,CUS_0x1037,2023-07-01,CUS_0x1037_2023_01_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
1,CUS_0x1069,2023-07-01,CUS_0x1069_2023_01_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
2,CUS_0x114a,2023-07-01,CUS_0x114a_2023_01_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
3,CUS_0x1184,2023-07-01,CUS_0x1184_2023_01_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
4,CUS_0x1297,2023-07-01,CUS_0x1297_2023_01_01,1,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4032,CUS_0xf2e,2024-01-01,CUS_0xf2e_2023_07_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
4033,CUS_0xf5d,2024-01-01,CUS_0xf5d_2023_07_01,1,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
4034,CUS_0xf8f,2024-01-01,CUS_0xf8f_2023_07_01,1,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN
4035,CUS_0xfaa,2024-01-01,CUS_0xfaa_2023_07_01,0,30dpd_6mob,None,None,NaN,NaN,NaN,...,NaN,NaN,NaN,None,None,NaN,None,None,NaN,NaN


In [17]:
from pyspark.sql import functions as F

l = labels_sdf.alias("l")
f = features_sdf.alias("f")

joined = l.join(
    f,
    (F.col("l.Customer_ID") == F.col("f.Customer_ID")) &
    (F.col("l.snapshot_date") == F.col("f.snapshot_date")),
    "left",
)

joined.select(
    F.count("*").alias("rows"),
    F.count(F.col("f.Customer_ID")).alias("matched_rows"),
).show()

+----+------------+
|rows|matched_rows|
+----+------------+
|4037|           0|
+----+------------+



In [18]:
print("labels:", labels_sdf.select("Customer_ID", "snapshot_date").dtypes)
print("features:", features_sdf.select("Customer_ID", "snapshot_date").dtypes)

labels_sdf.select("Customer_ID", "snapshot_date").show(10, truncate=False)
features_sdf.select("Customer_ID", "snapshot_date").show(10, truncate=False)

labels: [('Customer_ID', 'string'), ('snapshot_date', 'date')]
features: [('Customer_ID', 'string'), ('snapshot_date', 'date')]
+-----------+-------------+
|Customer_ID|snapshot_date|
+-----------+-------------+
|CUS_0x1037 |2023-07-01   |
|CUS_0x1069 |2023-07-01   |
|CUS_0x114a |2023-07-01   |
|CUS_0x1184 |2023-07-01   |
|CUS_0x1297 |2023-07-01   |
|CUS_0x12fb |2023-07-01   |
|CUS_0x1325 |2023-07-01   |
|CUS_0x1341 |2023-07-01   |
|CUS_0x1375 |2023-07-01   |
|CUS_0x13a8 |2023-07-01   |
+-----------+-------------+
only showing top 10 rows

+-----------+-------------+
|Customer_ID|snapshot_date|
+-----------+-------------+
|CUS_0x1048 |2024-02-01   |
|CUS_0x10c0 |2024-02-01   |
|CUS_0x115c |2024-02-01   |
|CUS_0x12b6 |2024-02-01   |
|CUS_0x12b8 |2024-02-01   |
|CUS_0x12ef |2024-02-01   |
|CUS_0x134d |2024-02-01   |
|CUS_0x13c2 |2024-02-01   |
|CUS_0x1414 |2024-02-01   |
|CUS_0x1470 |2024-02-01   |
+-----------+-------------+
only showing top 10 rows



In [19]:
l = labels_sdf.select("Customer_ID", "snapshot_date").distinct().alias("l")
f = features_sdf.select("Customer_ID", "snapshot_date").distinct().alias("f")

l.join(f, "Customer_ID", "inner").count(), \
l.join(f, "snapshot_date", "inner").count()

(4037, 1954922)